# Telco Churn — Customer Archetypes and the F1 Decision Layer

**Goal:** raise F1 using UMAP clustering, holding recall at or above 0.80.

**Why this notebook does not add clusters as a feature.** That was already tested in
this project and it failed everywhere: 7 clustering-derived feature arms (best
+0.0043, inside the ±0.01 noise band), per-cluster models for all five algorithms
(−0.0086 to −0.0823), cluster-as-feature (sign-flipping noise). The reason is
structural, not a tuning miss: **a cluster label is a deterministic function of `X`**,
so a model that already sees `X` gains no information from being told which cluster a
customer is in. Clustering on a feature *subset* does not escape this either — a
function of `S ⊆ X` is still a function of `X`.

**Where clustering could still pay: the decision layer.** The model emits one score
per customer and we apply one cutoff. Per-archetype cutoffs are a decision rule the
model cannot express. But there is a precise condition on when that helps:

> If the model is perfectly calibrated, the F1-optimal rule **is** a single global
> threshold on `p`, whatever the segment. So **per-archetype decisions help if and
> only if the model's calibration differs by archetype.**

That is measurable in seconds, so this notebook measures it before building anything
expensive. Phase 2 is a **gate**: if calibration is the same in every archetype, no
per-archetype decision rule can beat the global threshold, and the honest deliverable
is the evidence for that rather than an arm that scores +0.002 by luck.

| Phase | What it does |
|---|---|
| 0 | Baseline: threshold chosen out-of-fold under the recall ≥ 0.80 floor |
| 1 | Customer archetypes: UMAP + HDBSCAN on the full customer population |
| 2 | **The gate**: does calibration differ by archetype? |
| 3 | Only if the gate passes — per-archetype calibration / joint thresholds |

Plan: `Archetype_F1_Plan.md`. Data: `data/telco_churn_model.csv`.

In [1]:
# If needed:  %pip install -q -r requirements.txt
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import HDBSCAN
from sklearn.metrics import (f1_score, recall_score, precision_score, roc_auc_score,
                             average_precision_score, silhouette_score, log_loss)
from scipy.stats import chi2

import umap
import xgboost as xgb
import optuna

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- dataviz palette: reference instance, light surface (matches the other notebooks) ---
SURFACE, INK, SECONDARY, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#008300", "#e87ba4", "#eda100", "#1baf7a"]   # slots 1-5
DEEMPH = "#898781"

AXIS = dict(gridcolor=GRID, linecolor=BASELINE, tickfont=dict(color=MUTED),
            title_font=dict(color=MUTED))
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'


def base_layout(fig, title, height=440, **kw):
    fig.update_layout(
        title=dict(text=title, font=dict(color=INK, size=16)),
        plot_bgcolor=SURFACE, paper_bgcolor=SURFACE, font=dict(family=FONT),
        height=height, **kw)
    return fig


print("environment ready")

environment ready


## Phase 0 — The baseline, measured honestly

Same feature path and same Optuna parameters as `Telco_Churn_LR_XGBoost.ipynb`, so the
scores are comparable. Two things matter here:

- **The threshold is chosen on out-of-fold train predictions, never on test.** Picking
  it on test and then reporting test is how you report a number you cannot reproduce.
- **The floor is recall ≥ 0.80.** Raw F1 peaks near threshold 0.60 at recall 0.75 —
  that trades away ~29 churners per 1,409 customers for +0.014 F1. The floor keeps the
  business goal intact.

Every arm in this notebook is judged against this bar, not against the default-0.5 F1.

In [2]:
DATA_PATH = "data/telco_churn_model.csv"

df = pd.read_csv(DATA_PATH)
# "None" is in pandas' default NA list, so these two read back as NaN. Both mean
# "does not have it" -- a fact about the customer, not a missing value.
df["Offer"] = df["Offer"].fillna("No Offer")
df["Internet Type"] = df["Internet Type"].fillna("No Internet")
assert df.isna().sum().sum() == 0, "unexpected nulls"

TARGET = "Churn Value"
LEAKY = ["Satisfaction Score", "Customer Status", "Churn Label", "Churn Score",
         "CLTV", "Churn Category", "Churn Reason"]
assert not [c for c in df.columns if c in LEAKY], "leaky column reached the frame"
df = df.drop(columns=["Customer ID"])

# Feature engineering -- identical to the baseline notebook
d = df.copy()
d["tenure_bucket"] = pd.cut(d["Tenure in Months"], bins=[-1, 3, 12, 24, 48, np.inf],
                            labels=["0-3m", "4-12m", "13-24m", "25-48m", "48m+"]).astype(str)
COUNT_COLS = ["Phone Service", "Multiple Lines", "Online Security", "Online Backup",
              "Device Protection Plan", "Premium Tech Support", "Streaming TV",
              "Streaming Movies", "Streaming Music", "Unlimited Data"]
d["num_services"] = (d[COUNT_COLS] == "Yes").sum(axis=1)
assert (d["Tenure in Months"] > 0).all()
d["avg_monthly_spend"] = d["Total Charges"] / d["Tenure in Months"]
d["monthly_to_total_ratio"] = d["Monthly Charge"] / (d["Total Charges"] + 1.0)
d["has_offer"] = np.where(d["Offer"] == "No Offer", "No", "Yes")

X = d.drop(columns=[TARGET])
y = d[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (5634, 36)  Test: (1409, 36)


In [3]:
ordinal_levels = {
    "Contract":      ["Month-to-Month", "One Year", "Two Year"],
    "tenure_bucket": ["0-3m", "4-12m", "13-24m", "25-48m", "48m+"],
    "Offer":         ["No Offer", "Offer A", "Offer B", "Offer C", "Offer D", "Offer E"],
}
obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
ord_cols = [c for c in ordinal_levels if c in obj_cols]
cat_cols = [c for c in obj_cols if c not in ord_cols]
num_cols = X.select_dtypes(include=["number"]).columns.tolist()

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("ord", Pipeline([("encode", OrdinalEncoder(categories=[ordinal_levels[c] for c in ord_cols])),
                      ("scale", StandardScaler())]), ord_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_cols),
]).fit(X_train, y_train)            # fit on TRAIN only

X_train_enc = preprocess.transform(X_train)
X_test_enc = preprocess.transform(X_test)
y_tr, y_te = y_train.values, y_test.values
scale_pos_weight = float((y_tr == 0).sum() / (y_tr == 1).sum())
print(f"encoded: {X_train_enc.shape[1]} features   scale_pos_weight {scale_pos_weight:.3f}")

encoded: 41 features   scale_pos_weight 2.769


In [4]:
# Optuna params carried over from Telco_Churn_LR_XGBoost.ipynb (CV ROC-AUC 0.9136).
# Re-tuning here would only add noise to a comparison that is about the decision layer.
BEST_PARAMS = {
    "n_estimators": 692, "max_depth": 5, "learning_rate": 0.017983890954223557,
    "subsample": 0.9273851446364887, "colsample_bytree": 0.9452482414080953,
    "min_child_weight": 8, "gamma": 4.447876955491479,
    "reg_alpha": 0.0022945430578750975, "reg_lambda": 0.5316388882904249,
}
model = xgb.XGBClassifier(**BEST_PARAMS, objective="binary:logistic", eval_metric="auc",
                          tree_method="hist", scale_pos_weight=scale_pos_weight,
                          random_state=RANDOM_STATE, n_jobs=-1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Out-of-fold train predictions: every score here comes from a model that did not
# see that row. This is what the threshold and the gate are allowed to look at.
oof_proba = cross_val_predict(model, X_train_enc, y_tr, cv=cv,
                              method="predict_proba", n_jobs=-1)[:, 1]

model.fit(X_train_enc, y_tr)
test_proba = model.predict_proba(X_test_enc)[:, 1]

print(f"OOF  ROC-AUC {roc_auc_score(y_tr, oof_proba):.4f}   PR-AUC {average_precision_score(y_tr, oof_proba):.4f}")
print(f"TEST ROC-AUC {roc_auc_score(y_te, test_proba):.4f}   PR-AUC {average_precision_score(y_te, test_proba):.4f}")

OOF  ROC-AUC 0.9132   PR-AUC 0.8006
TEST ROC-AUC 0.9155   PR-AUC 0.8032


In [5]:
RECALL_FLOOR = 0.80

grid = np.linspace(0.05, 0.95, 181)
feasible = [t for t in grid if recall_score(y_tr, (oof_proba >= t).astype(int)) >= RECALL_FLOOR]
assert feasible, f"no threshold reaches recall {RECALL_FLOOR} out-of-fold"
T_GLOBAL = max(feasible, key=lambda t: f1_score(y_tr, (oof_proba >= t).astype(int)))

pred_base = (test_proba >= T_GLOBAL).astype(int)
BASE_F1 = f1_score(y_te, pred_base)

print(f"threshold chosen out-of-fold under recall >= {RECALL_FLOOR}:  t* = {T_GLOBAL:.3f}")
print(f"  OOF   recall {recall_score(y_tr, (oof_proba >= T_GLOBAL).astype(int)):.4f}")
print()
print("=== PHASE 0 BASELINE (the bar every arm must beat) ===")
print(f"  TEST  F1 {BASE_F1:.4f}   recall {recall_score(y_te, pred_base):.4f}   "
      f"precision {precision_score(y_te, pred_base):.4f}")
print()
print("  for reference, the default cutoff:")
p05 = (test_proba >= 0.5).astype(int)
print(f"  t=0.5 TEST  F1 {f1_score(y_te, p05):.4f}   recall {recall_score(y_te, p05):.4f}   "
      f"precision {precision_score(y_te, p05):.4f}")

threshold chosen out-of-fold under recall >= 0.8:  t* = 0.545
  OOF   recall 0.8094

=== PHASE 0 BASELINE (the bar every arm must beat) ===
  TEST  F1 0.7189   recall 0.7968   precision 0.6549

  for reference, the default cutoff:
  t=0.5 TEST  F1 0.7166   recall 0.8316   precision 0.6296


## Phase 1 — Customer archetypes

### Features are selected by their association with churn

Earlier feature sets kept columns because they were *behavioural*. That produced clusters
that would not form: every hyperparameter combination in a manual sweep collapsed to the
same degenerate 2-way internet / no-internet split (Cramér's V = 0.998 against
`Internet Type`, 0% noise), because a handful of churn-irrelevant columns —
`Avg Monthly GB Download`, `num_streaming`, `Latitude`, `Longitude`, `Population`,
`Gender` — dominated the distance metric while telling us nothing about churn.

So the rule is now explicit: **a column earns its place in the clustering space by its
association with `Churn Value`.** Measured on **train only** (measuring on the full
dataset would leak the test set into the feature choice), with:

- **point-biserial |r|** for numeric columns,
- **Cramér's V** for categorical columns.

Both live on 0–1 and mean "how strongly does this column move with churn".

### The cut is at 0.10, and the data chose it

The ranked associations have a natural gap:

| | feature | association |
|---|---|---|
| ✅ | `Internet Type` | 0.306 |
| ✅ | `Number of Referrals` | 0.285 |
| ✅ | `Payment Method` | 0.217 |
| ✅ | `Number of Dependents` | 0.214 |
| ✅ | `Paperless Billing` | 0.196 |
| ✅ | `Monthly Charge` | 0.190 |
| ✅ | `num_protection` | 0.176 |
| ✅ | `Unlimited Data` | 0.165 |
| ✅ | `Married` | 0.150 |
| ✅ | `Age` | 0.113 |
| | **— gap, 1.6× drop —** | |
| ❌ | `num_streaming` | 0.070 |
| ❌ | `Avg Monthly GB Download` | 0.049 |
| ❌ | `had_overage` | 0.048 |
| ❌ | `Population` | 0.046 |
| ❌ | `Multiple Lines` | 0.036 |
| ❌ | `Latitude` | 0.034 |
| ❌ | `had_refund` | 0.028 |
| ❌ | `Longitude` | 0.023 |
| ❌ | `Avg Monthly Long Distance Charges` | 0.007 |
| ❌ | `Gender` | 0.001 |

`Age` at 0.113 and `num_streaming` at 0.070 are separated by a real 1.6× step, so the
0.10 threshold falls in a gap rather than slicing through a continuum. The selection is
**computed in the cell below, not hardcoded** — change `CHURN_ASSOC_MIN` and the space
rebuilds itself.

### Pruning for churn also fixed the cluster-formation problem

This was not the goal, but it is why the clusters now form. `Avg Monthly GB Download` and
`num_streaming` both go to exactly 0 for every no-internet customer. Together with
`Internet Type` they made no-internet customers sit ~5 dimensions away from everyone
else — a chasm so wide that HDBSCAN spent its whole budget on it and found nothing else.
Dropping them (for being churn-irrelevant) removed two of those dimensions, and the
degenerate split stopped being the only structure in the data.

### The honest caveat: this is now supervised

Selecting clustering features by their correlation with the target means these archetypes
are **not** an unsupervised discovery — they are "cluster inside a churn-relevant
subspace". That is a legitimate technique and it is the right one for the retention use
case, but two consequences follow:

1. The selection must be **train-only**, which it is. Otherwise the test set influences
   the feature space and every downstream number is optimistic.
2. It makes the Phase 2 gate **less** likely to pass, not more. Archetypes aligned to
   churn are aligned to what the model already scores well — the last place a calibration
   gap can hide. The gate has already failed twice; this is not expected to rescue it.
   **The goal here is archetype quality for the SHAP → LLM retention layer, not F1.**

Still excluded: the lifecycle axis — `Contract` (0.458), `Tenure in Months` (0.354),
`Total Charges` (0.201) and the tenure-derived ratios. These are the *strongest* churn
signals in the file, so excluding them costs real churn separation. They stay out because
archetypes built on them would simply restate the model's own risk ranking, which is both
useless as a segment ("high-risk customers churn") and the least likely place to find a
calibration gap.

### On the objective

`silhouette × (1 − noise)` with the guards **on**. Bare silhouette is not usable here: it
is maximised by finding one enormous gap — exactly how the degenerate split scored 0.7996
in an earlier run. Silhouette on a UMAP embedding also scores *the embedding*, not the
clustering, so a bootstrap-ARI stability check follows the search as an independent read.

In [6]:
# --- Archetype features, derived here so the MODEL's feature space stays untouched.
# Adding these to X would change X_train_enc and move the Phase 0 baseline, which has to
# stay comparable with Telco_Churn_LR_XGBoost.ipynb.
STREAMING = ["Streaming TV", "Streaming Movies", "Streaming Music"]
PROTECTION = ["Online Security", "Online Backup", "Device Protection Plan",
              "Premium Tech Support"]


def archetype_frame(Xf):
    """Every CANDIDATE column. What survives is decided below, by association with churn."""
    f = pd.DataFrame(index=Xf.index)
    # Counts instead of raw flags: Streaming Movies <-> Music correlate 0.849, so keeping
    # both would double-weight "is a streamer" purely by column count.
    f["num_streaming"] = (Xf[STREAMING] == "Yes").sum(axis=1)         # 0-3
    f["num_protection"] = (Xf[PROTECTION] == "Yes").sum(axis=1)       # 0-4
    # Zero-inflated -> binary. Left continuous these are ~90% zeros and act as noise.
    f["had_refund"] = np.where(Xf["Total Refunds"] > 0, "Yes", "No")
    f["had_overage"] = np.where(Xf["Total Extra Data Charges"] > 0, "Yes", "No")
    for c in ["Avg Monthly GB Download", "Avg Monthly Long Distance Charges",
              "Monthly Charge", "Age", "Number of Referrals", "Number of Dependents",
              "Internet Type", "Multiple Lines", "Unlimited Data", "Married",
              "Payment Method", "Paperless Billing", "Gender", "Latitude", "Longitude",
              "Population"]:
        f[c] = Xf[c].values
    return f


CANDIDATE_NUM = ["Avg Monthly GB Download", "Avg Monthly Long Distance Charges",
                 "Monthly Charge", "Age", "Number of Referrals", "Number of Dependents",
                 "num_streaming", "num_protection", "Latitude", "Longitude", "Population"]
CANDIDATE_CAT = ["Internet Type", "Multiple Lines", "Unlimited Data", "Married",
                 "Payment Method", "Paperless Billing", "had_refund", "had_overage",
                 "Gender"]

# The lifecycle axis: the strongest churn signals in the file, deliberately withheld.
# Archetypes built on them would restate the model's own risk ranking -- useless as a
# segment, and the least likely place a calibration gap could hide.
EXCLUDED_LIFECYCLE = ["Contract", "Tenure in Months", "tenure_bucket", "Offer",
                      "has_offer", "avg_monthly_spend", "monthly_to_total_ratio",
                      "Total Charges"]

A_train, A_test = archetype_frame(X_train), archetype_frame(X_test)


def cramers_v(a, b):
    ct = pd.crosstab(a, b).values
    n = ct.sum()
    exp = np.outer(ct.sum(1), ct.sum(0)) / n
    stat = ((ct - exp) ** 2 / exp).sum()
    r, k = ct.shape
    return float(np.sqrt((stat / n) / max(min(r - 1, k - 1), 1)))


def churn_assoc(values):
    """0-1 strength of association with churn. Point-biserial |r| for numeric,
    Cramer's V for categorical -- both answer 'does this move with churn'."""
    if pd.api.types.is_numeric_dtype(values):
        return float(abs(np.corrcoef(values.astype(float), y_tr)[0, 1]))
    return cramers_v(values, y_tr)


# --- Select on TRAIN ONLY. Measuring association on the full frame would let the test
# set choose the feature space, and every number after that would be optimistic.
CHURN_ASSOC_MIN = 0.10        # the gap sits between Age (0.113) and num_streaming (0.070)

assoc_tbl = (pd.DataFrame(
    [{"feature": c,
      "churn_assoc": churn_assoc(A_train[c].values),
      "measure": "point-biserial |r|" if c in CANDIDATE_NUM else "Cramer's V"}
     for c in CANDIDATE_NUM + CANDIDATE_CAT])
    .sort_values("churn_assoc", ascending=False).reset_index(drop=True))
assoc_tbl["keep"] = np.where(assoc_tbl["churn_assoc"] >= CHURN_ASSOC_MIN, "KEEP", "drop")

CLUSTER_NUM = [c for c in CANDIDATE_NUM
               if assoc_tbl.set_index("feature").loc[c, "churn_assoc"] >= CHURN_ASSOC_MIN]
CLUSTER_CAT = [c for c in CANDIDATE_CAT
               if assoc_tbl.set_index("feature").loc[c, "churn_assoc"] >= CHURN_ASSOC_MIN]

_space = set(CLUSTER_NUM + CLUSTER_CAT)
assert not (_space & set(EXCLUDED_LIFECYCLE)), "a lifecycle column leaked into the space"
assert TARGET not in _space, "target in the clustering space"
assert _space, f"no feature cleared CHURN_ASSOC_MIN={CHURN_ASSOC_MIN}"

print(f"Feature selection by churn association (TRAIN only, cut at {CHURN_ASSOC_MIN})\n")
print(assoc_tbl.to_string(index=False, formatters={"churn_assoc": "{:.4f}".format}))
print(f"\nkept {len(_space)} of {len(CANDIDATE_NUM) + len(CANDIDATE_CAT)}: "
      f"{len(CLUSTER_NUM)} numeric + {len(CLUSTER_CAT)} categorical")

cluster_prep = ColumnTransformer([
    ("num", StandardScaler(), CLUSTER_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), CLUSTER_CAT),
]).fit(A_train)                        # TRAIN only

Bh_train = cluster_prep.transform(A_train)
Bh_test = cluster_prep.transform(A_test)
print(f"\narchetype space: {Bh_train.shape[1]} dims "
      f"(the model itself sees {X_train_enc.shape[1]})")
print(f"clustering all {len(A_train)} customers together (no segmentation)")

Feature selection by churn association (TRAIN only, cut at 0.1)

                          feature churn_assoc            measure keep
                    Internet Type      0.3063         Cramer's V KEEP
              Number of Referrals      0.2854 point-biserial |r| KEEP
                   Payment Method      0.2168         Cramer's V KEEP
             Number of Dependents      0.2135 point-biserial |r| KEEP
                Paperless Billing      0.1958         Cramer's V KEEP
                   Monthly Charge      0.1904 point-biserial |r| KEEP
                   num_protection      0.1761 point-biserial |r| KEEP
                   Unlimited Data      0.1647         Cramer's V KEEP
                          Married      0.1502         Cramer's V KEEP
                              Age      0.1129 point-biserial |r| KEEP
                    num_streaming      0.0696 point-biserial |r| drop
          Avg Monthly GB Download      0.0487 point-biserial |r| drop
                      had

In [7]:
N_CLUSTER_TRIALS = 40      # was 25 -- too thin for 4 hyperparameters; a previous run
                           # failed to find any valid configuration at all
NOISE_CAP = 0.35
# k=2 is excluded deliberately: on the churn-relevant space a manual sweep never
# produced it (minimum was 4), and the only 2-cluster solution this data offers is the
# degenerate internet / no-internet split. If a run ever needs k=2 to find anything,
# that is a signal the feature space regressed, not a range to widen.
CLUSTER_RANGE = (3, 6)
MIN_ARCHETYPE = 400        # below this a per-archetype threshold is fitting noise


def cluster_objective(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 15, 80)
    min_dist = trial.suggest_float("min_dist", 0.0, 0.5)
    mcs = trial.suggest_int("min_cluster_size", MIN_ARCHETYPE, 900)
    ms = trial.suggest_int("min_samples", 5, 50)

    emb = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=2,
                    random_state=RANDOM_STATE).fit_transform(Bh_train)
    labels = HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit_predict(emb)

    k = len(set(labels) - {-1})
    noise = float((labels == -1).mean())
    trial.set_user_attr("k", k)
    trial.set_user_attr("noise", noise)

    # Guards. Bare silhouette is not a usable objective: it is maximised by finding one
    # enormous gap (that is how a degenerate internet/no-internet split scored 0.7996),
    # or by fragmenting, or by calling most customers noise. All three score well and
    # mean nothing.
    if not (CLUSTER_RANGE[0] <= k <= CLUSTER_RANGE[1]):
        return -1.0
    if noise > NOISE_CAP:
        return -1.0

    keep = labels != -1
    sil = silhouette_score(emb[keep], labels[keep])
    # return sil
    return sil * (1.0 - noise)         # discarding hard customers is not free


cluster_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
cluster_study.optimize(cluster_objective, n_trials=N_CLUSTER_TRIALS, show_progress_bar=False)

# -1.0 is the rejection sentinel. Without this check, a study in which EVERY trial was
# rejected still hands back a best_params, and the notebook carries on with a
# configuration that failed the guards -- which is exactly what happened before.
assert cluster_study.best_value > 0, (
    f"no configuration satisfied the guards in {N_CLUSTER_TRIALS} trials "
    f"({CLUSTER_RANGE[0]}-{CLUSTER_RANGE[1]} clusters, noise <= {NOISE_CAP:.0%}). "
    f"Widen the search or revisit the feature space -- do NOT proceed on a rejected trial.")

bp = cluster_study.best_params
n_valid = sum(1 for t in cluster_study.trials if t.value is not None and t.value > 0)
print(f"{n_valid} of {N_CLUSTER_TRIALS} trials passed the guards")
print("best guarded silhouette:", round(cluster_study.best_value, 4))
print("  clusters:", cluster_study.best_trial.user_attrs["k"],
      " noise:", f'{cluster_study.best_trial.user_attrs["noise"]:.1%}')
print("  params:", bp)

32 of 40 trials passed the guards
best guarded silhouette: 0.6406
  clusters: 4  noise: 0.0%
  params: {'n_neighbors': 76, 'min_dist': 0.0032760775680389267, 'min_cluster_size': 790, 'min_samples': 9}


In [8]:
# Refit the winning configuration; assign test by nearest neighbours in the embedding
# (HDBSCAN has no predict). Everything is fit on TRAIN only.
umap_model = umap.UMAP(n_neighbors=bp["n_neighbors"], min_dist=bp["min_dist"],
                       n_components=2, random_state=RANDOM_STATE).fit(Bh_train)
emb_train = umap_model.embedding_
emb_test = umap_model.transform(Bh_test)

clu_train = HDBSCAN(min_cluster_size=bp["min_cluster_size"],
                    min_samples=bp["min_samples"]).fit_predict(emb_train)
_m = clu_train != -1
clu_test = KNeighborsClassifier(n_neighbors=15).fit(emb_train[_m],
                                                    clu_train[_m]).predict(emb_test)

ARCHETYPES = sorted(set(clu_train) - {-1})
assert all((clu_train == c).sum() >= MIN_ARCHETYPE for c in ARCHETYPES), \
    "an archetype is too small for its calibration estimate to mean anything"

NAMES = {c: f"Archetype {c}" for c in ARCHETYPES}

profile = pd.DataFrame([{
    "archetype": NAMES[c],
    "n_train": int((clu_train == c).sum()),
    "share": (clu_train == c).mean(),
    "churn_rate": y_tr[clu_train == c].mean(),
} for c in ARCHETYPES]).set_index("archetype")

print(f"{len(ARCHETYPES)} archetypes, noise {(clu_train == -1).mean():.1%}, "
      f"overall churn {y_tr.mean():.1%}\n")
print(profile.to_string(formatters={"share": "{:.1%}".format,
                                    "churn_rate": "{:.1%}".format}))

4 archetypes, noise 0.0%, overall churn 26.5%

             n_train share churn_rate
archetype                            
Archetype 0     1227 21.8%       7.5%
Archetype 1     2206 39.2%      18.5%
Archetype 2     1281 22.7%      38.5%
Archetype 3      920 16.3%      54.5%


In [9]:
# Are these archetypes real, or an artefact of this particular sample?
# Silhouette scores the UMAP embedding, not the clustering -- it rewards whatever UMAP
# happened to pull apart. Stability asks the question we actually care about: resample,
# re-cluster from scratch, and see whether the same customers land together again.
from sklearn.metrics import adjusted_rand_score

N_BOOT, FRAC = 5, 0.8
rng = np.random.default_rng(RANDOM_STATE)
boot = []
for b in range(N_BOOT):
    idx = rng.choice(len(Bh_train), size=int(FRAC * len(Bh_train)), replace=False)
    emb_b = umap.UMAP(n_neighbors=bp["n_neighbors"], min_dist=bp["min_dist"],
                      n_components=2,
                      random_state=RANDOM_STATE + b).fit_transform(Bh_train[idx])
    lab_b = HDBSCAN(min_cluster_size=int(bp["min_cluster_size"] * FRAC),
                    min_samples=bp["min_samples"]).fit_predict(emb_b)
    boot.append(adjusted_rand_score(clu_train[idx], lab_b))

STABILITY = float(np.mean(boot))
print(f"bootstrap stability — {N_BOOT} resamples at {FRAC:.0%}, re-clustered from scratch")
print(f"  per-run ARI : {[round(s, 3) for s in boot]}")
print(f"  mean ARI    : {STABILITY:.3f}   (1.0 = identical partitions, 0.0 = chance)")
print(f"\n  {'STABLE' if STABILITY >= 0.70 else 'UNSTABLE'} — "
      + ("the same customers cluster together across resamples"
         if STABILITY >= 0.70 else
         "the archetypes move when the sample moves; treat them as indicative only"))

bootstrap stability — 5 resamples at 80%, re-clustered from scratch
  per-run ARI : [0.991, 0.84, 0.842, 0.996, 0.995]
  mean ARI    : 0.933   (1.0 = identical partitions, 0.0 = chance)

  STABLE — the same customers cluster together across resamples


In [10]:
# Are these their own thing, or a restatement of something the model already leans on?
# cramers_v is defined in the feature-selection cell above.
m = clu_train != -1
assoc = {c: cramers_v(clu_train[m], X_train[c].values[m])
         for c in ["Contract", "tenure_bucket", "Internet Type", "Payment Method",
                   "num_services"]}
print("Cramer's V — archetype vs feature (0 = independent, 1 = identical)")
for k_, v in sorted(assoc.items(), key=lambda kv: -kv[1]):
    print(f"  {k_:16s} {v:.3f}")

# Archetypes must not be a Contract proxy, or they carve the space along the axis the
# model already handles best. Internet Type is reported rather than asserted: it is the
# single strongest churn signal we allow in, so it SHOULD rank high -- but at V ~= 1.0 it
# would mean the degenerate chasm re-formed and the archetypes are just "has internet".
assert assoc["Contract"] < 0.35, "archetypes are a Contract proxy — rebuild Phase 1"
assert assoc["Internet Type"] < 0.95, \
    "archetypes collapsed back onto Internet Type — the chasm re-formed"
print(f"\nPASS: not a Contract proxy (V = {assoc['Contract']:.3f}); "
      f"not an Internet Type restatement (V = {assoc['Internet Type']:.3f})")

Cramer's V — archetype vs feature (0 = independent, 1 = identical)
  Internet Type    0.769
  num_services     0.506
  Payment Method   0.296
  Contract         0.200
  tenure_bucket    0.124

PASS: not a Contract proxy (V = 0.200); not an Internet Type restatement (V = 0.769)


### The archetypes, in the space they were clustered in

In [11]:
COLOR = {c: SERIES[i % len(SERIES)] for i, c in enumerate(ARCHETYPES)}
COLOR[-1] = DEEMPH

fig = go.Figure()
for c in ARCHETYPES + ([-1] if (clu_train == -1).any() else []):
    m = clu_train == c
    name = NAMES.get(c, "Unclustered")
    fig.add_trace(go.Scattergl(
        x=emb_train[m, 0], y=emb_train[m, 1], mode="markers", name=name,
        marker=dict(color=COLOR[c], size=4, opacity=0.55, line=dict(width=0)),
        hovertemplate=f"<b>{name}</b><br>churn rate {y_tr[m].mean():.1%}<extra></extra>",
    ))

# Direct labels at the centroids — the palette's lighter slots sit under 3:1 against
# the surface, so identity never rests on the legend swatch alone.
for c in ARCHETYPES:
    m = clu_train == c
    fig.add_annotation(x=emb_train[m, 0].mean(), y=emb_train[m, 1].mean(),
                       text=f"<b>{NAMES[c]}</b><br>{y_tr[m].mean():.0%} churn",
                       showarrow=False, font=dict(color=INK, size=11),
                       bgcolor="rgba(252,252,251,0.78)", borderpad=3)

base_layout(fig, "Customer archetypes — UMAP embedding (all customers)", height=560,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                        font=dict(color=INK)),
            margin=dict(t=70, r=20, b=50, l=60))
fig.update_xaxes(title="UMAP 1", **AXIS)
fig.update_yaxes(title="UMAP 2", **AXIS)
fig.show()

### Churn rate by archetype — the seductive chart

The archetypes separate churn hard, from single digits to near half. It is tempting to
read that as "clustering found the churn structure, so it must help."

It does not follow, and this is the trap the whole notebook is built to avoid. The
model **already knows** everything the archetypes know — they were built from features
it can see. A spread in churn rate across archetypes shows the clusters are *real*, not
that they are *informative on top of the model*. Whether they can help is Phase 2's
question, and this chart cannot answer it.

In [12]:
order = profile.sort_values("churn_rate").index.tolist()
vals = [profile.loc[c, "churn_rate"] for c in order]
label_to_c = {NAMES[c]: c for c in ARCHETYPES}

fig = go.Figure(go.Bar(
    x=order, y=vals,
    marker=dict(color=[COLOR[label_to_c[c]] for c in order], cornerradius=4), width=0.5,
    text=[f"{v:.1%}" for v in vals], textposition="outside",
    textfont=dict(color=SECONDARY, size=11),
    customdata=[profile.loc[c, "n_train"] for c in order],
    hovertemplate="<b>%{x}</b><br>churn %{y:.1%}<br>%{customdata} customers<extra></extra>",
))
fig.add_hline(y=y_tr.mean(), line=dict(color=INK, width=2, dash="dash"),
              annotation_text=f"overall {y_tr.mean():.1%}", annotation_position="top left",
              annotation_font=dict(color=MUTED, size=11))

base_layout(fig, "Churn rate by archetype (train)", height=420,
            showlegend=False, margin=dict(t=60, r=20, b=50, l=60))
fig.update_xaxes(showgrid=False, linecolor=BASELINE, tickfont=dict(color=SECONDARY))
fig.update_yaxes(title="Churn rate", tickformat=".0%", range=[0, max(vals) * 1.25], **AXIS)
fig.show()

In [13]:
# Archetypes you can describe in a sentence -- the point of the whole exercise.
desc = A_train.copy()
desc["_a"] = [NAMES[c] if c != -1 else "Unclustered" for c in clu_train]
num_view = desc.groupby("_a")[[c for c in CLUSTER_NUM]].mean().round(2)
cat_view = pd.DataFrame({c: desc.groupby("_a")[c].agg(lambda s: s.mode()[0])
                         for c in CLUSTER_CAT})
churn_view = pd.Series({NAMES[c] if c != -1 else "Unclustered": y_tr[clu_train == c].mean()
                        for c in sorted(set(clu_train))}, name="churn")

print("What each archetype actually is (selected features only)\n")
print(num_view.join(churn_view).to_string(formatters={"churn": "{:.1%}".format}))
print()
print(cat_view.to_string())

What each archetype actually is (selected features only)

             Monthly Charge    Age  Number of Referrals  Number of Dependents  num_protection churn
_a                                                                                                 
Archetype 0           21.14  42.67                 2.12                  0.77            0.00  7.5%
Archetype 1           62.85  44.33                 2.29                  0.71            1.87 18.5%
Archetype 2           93.42  51.28                 2.66                  0.09            1.61 38.5%
Archetype 3           88.41  50.48                 0.00                  0.02            1.03 54.5%

            Internet Type Unlimited Data Married   Payment Method Paperless Billing
_a                                                                                 
Archetype 0   No Internet             No      No      Credit Card                No
Archetype 1           DSL            Yes     Yes  Bank Withdrawal               Yes
Arche

## Phase 3 — Predicting churn probability with five models (archetype label as a feature)

This phase does something the rest of the notebook argues *against*: it adds the
archetype label to the model's feature set. That is a deliberate, requested experiment,
so it is worth being clear-eyed about it. The notebook's structural point still stands —
the archetype is a deterministic function of `X`, so a model that already sees `X` learns
little from being told the cluster. The value here is not "clustering lifts F1"; it is a
clean, comparable **five-model bake-off** that predicts per-customer churn probability,
with the archetype available as one more column.

**What is trained**

| Model | Tuning |
|---|---|
| Logistic Regression | defaults, `class_weight="balanced"` |
| Decision Tree | shallow, leaf-regularised, `class_weight="balanced"` |
| Random Forest | 400 trees, `class_weight="balanced"` |
| **LightGBM** | Optuna, **50 trials**, objective = maximise recall s.t. precision floor |
| **XGBoost** | Optuna, **50 trials**, same objective |

**The tuning objective.** Maximising recall *alone* is degenerate — predict that everyone
churns and recall is exactly 1.0. So recall is maximised **subject to a minimum precision
floor** (`MIN_PRECISION`), which is the operating point a retention team would actually
run. Every score inside the objective is out-of-fold, so the hyperparameters are not
chosen on data the trial's model has seen.

**The feature matrix.** The engineered, leak-checked feature path from Phase 0 (41
columns) plus a one-hot of the archetype label (4 columns). The archetype is fit on train
only; the test set's archetype comes from the Phase 1 KNN assignment (`clu_test`) — the
test data is never clustered on its own.

In [14]:
import lightgbm as lgb
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, confusion_matrix, roc_curve


def to_dense(a):
    return a.toarray() if hasattr(a, "toarray") else np.asarray(a)


# One-hot the archetype label. categories are pinned to the TRAIN archetypes so train and
# test always encode to the same columns, even if a test archetype happened to be empty.
clu_ohe = OneHotEncoder(categories=[ARCHETYPES], handle_unknown="ignore",
                        sparse_output=False)
clu_train_oh = clu_ohe.fit_transform(np.asarray(clu_train).reshape(-1, 1))
clu_test_oh = clu_ohe.transform(np.asarray(clu_test).reshape(-1, 1))

BASE_FEATURES = list(preprocess.get_feature_names_out())
CLUSTER_FEATURES = [f"cluster={NAMES[c]}" for c in ARCHETYPES]
MODEL_FEATURES = BASE_FEATURES + CLUSTER_FEATURES

# The engineered path (Phase 0, leak-checked) + the archetype one-hot.
Xtr_model = np.hstack([to_dense(X_train_enc), clu_train_oh])
Xte_model = np.hstack([to_dense(X_test_enc), clu_test_oh])
assert Xtr_model.shape[1] == len(MODEL_FEATURES), "feature-name count != matrix width"

print(f"model matrix: {Xtr_model.shape[1]} features "
      f"({len(BASE_FEATURES)} engineered + {len(CLUSTER_FEATURES)} archetype)")
print("archetype columns:", CLUSTER_FEATURES)

model matrix: 45 features (41 engineered + 4 archetype)
archetype columns: ['cluster=Archetype 0', 'cluster=Archetype 1', 'cluster=Archetype 2', 'cluster=Archetype 3']


In [15]:
N_TRIALS = 50

# Maximising recall alone is degenerate: predict everyone churns and recall = 1.0. So the
# objective maximises recall while precision stays at or above this floor -- the operating
# point retention will actually run. Set it to the lowest precision you are willing to
# ship; nothing downstream is hardcoded to a particular value.
MIN_PRECISION = 0.55

# 3-fold inside tuning keeps 50 trials x 2 models affordable; the final comparison below
# uses the notebook's 5-fold `cv`.
tune_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)


def recall_at_precision_floor(y_true, proba, floor):
    """Highest recall achievable while precision stays >= floor. Returns 0.0 when the
    floor is unreachable at every threshold, so those trials are penalised, not rewarded."""
    prec, rec, _ = precision_recall_curve(y_true, proba)
    ok = prec[:-1] >= floor            # drop the (P=1, R=0) endpoint, which has no threshold
    return float(rec[:-1][ok].max()) if ok.any() else 0.0


def operating_threshold(y_true, proba, floor):
    """The cut that realises recall_at_precision_floor: the lowest threshold whose
    precision still clears the floor (hence the largest feasible recall)."""
    prec, rec, thr = precision_recall_curve(y_true, proba)
    prec, rec = prec[:-1], rec[:-1]
    ok = prec >= floor
    if not ok.any():
        return 0.5
    return float(thr[np.argmax(np.where(ok, rec, -1.0))])


print(f"tuning objective: maximise recall s.t. precision >= {MIN_PRECISION}  "
      f"({N_TRIALS} trials, {tune_cv.get_n_splits()}-fold out-of-fold)")

tuning objective: maximise recall s.t. precision >= 0.55  (50 trials, 3-fold out-of-fold)


In [16]:
# --- XGBoost: 50 Optuna trials, maximising OOF recall at the precision floor ---
def build_xgb(p):
    return xgb.XGBClassifier(**p, objective="binary:logistic", eval_metric="aucpr",
                             tree_method="hist", scale_pos_weight=scale_pos_weight,
                             random_state=RANDOM_STATE, n_jobs=-1)


def xgb_objective(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", 200, 900),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
    )
    oof = cross_val_predict(build_xgb(p), Xtr_model, y_tr, cv=tune_cv,
                            method="predict_proba", n_jobs=-1)[:, 1]
    return recall_at_precision_floor(y_tr, oof, MIN_PRECISION)


xgb_study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=False)
xgb_best = xgb_study.best_params
print(f"XGBoost  best OOF recall @ precision>={MIN_PRECISION}: {xgb_study.best_value:.4f}")
print("  params:", xgb_best)

XGBoost  best OOF recall @ precision>=0.55: 0.9157
  params: {'n_estimators': 450, 'max_depth': 4, 'learning_rate': 0.022096526145513846, 'subsample': 0.6563696899899051, 'colsample_bytree': 0.9208787923016158, 'min_child_weight': 1, 'gamma': 4.9344346830025865, 'reg_alpha': 0.7186380986530759, 'reg_lambda': 0.005433045540798127}


In [17]:
# --- LightGBM: 50 Optuna trials, same objective ---
def build_lgbm(p):
    return lgb.LGBMClassifier(**p, objective="binary", scale_pos_weight=scale_pos_weight,
                              random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)


def lgbm_objective(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", 200, 900),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 80),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
    )
    oof = cross_val_predict(build_lgbm(p), Xtr_model, y_tr, cv=tune_cv,
                            method="predict_proba", n_jobs=-1)[:, 1]
    return recall_at_precision_floor(y_tr, oof, MIN_PRECISION)


lgbm_study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS, show_progress_bar=False)
lgbm_best = lgbm_study.best_params
print(f"LightGBM best OOF recall @ precision>={MIN_PRECISION}: {lgbm_study.best_value:.4f}")
print("  params:", lgbm_best)

LightGBM best OOF recall @ precision>=0.55: 0.9157
  params: {'n_estimators': 773, 'num_leaves': 129, 'max_depth': 4, 'learning_rate': 0.009779309302827156, 'subsample': 0.6634104471957729, 'colsample_bytree': 0.6901978638673252, 'min_child_samples': 22, 'reg_alpha': 0.027418619841709353, 'reg_lambda': 0.0011187924287119225}


In [18]:
# --- The five models. LR/DT/RF use class_weight="balanced" so they also lean toward
# recall; LightGBM/XGBoost carry the Optuna-tuned params from the two cells above.
MODELS = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced",
                                              random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                            class_weight="balanced",
                                            random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=5,
                                            class_weight="balanced", n_jobs=-1,
                                            random_state=RANDOM_STATE),
    "LightGBM": build_lgbm(lgbm_best),
    "XGBoost": build_xgb(xgb_best),
}

fitted, oof_proba_m, test_proba_m, thr_m = {}, {}, {}, {}
for name, est in MODELS.items():
    # OOF probabilities on train pick the operating threshold -- test is never used for it.
    oof = cross_val_predict(est, Xtr_model, y_tr, cv=cv,
                            method="predict_proba", n_jobs=-1)[:, 1]
    est.fit(Xtr_model, y_tr)
    oof_proba_m[name] = oof
    test_proba_m[name] = est.predict_proba(Xte_model)[:, 1]
    thr_m[name] = operating_threshold(y_tr, oof, MIN_PRECISION)
    fitted[name] = est
    print(f"{name:20s} threshold {thr_m[name]:.3f}  "
          f"(OOF recall {recall_score(y_tr, (oof >= thr_m[name]).astype(int)):.3f}, "
          f"precision {precision_score(y_tr, (oof >= thr_m[name]).astype(int)):.3f})")

Logistic Regression  threshold 0.430  (OOF recall 0.878, precision 0.550)
Decision Tree        threshold 0.444  (OOF recall 0.844, precision 0.552)
Random Forest        threshold 0.375  (OOF recall 0.878, precision 0.550)
LightGBM             threshold 0.360  (OOF recall 0.910, precision 0.550)
XGBoost              threshold 0.335  (OOF recall 0.910, precision 0.550)


### Predicting the churn probability of a customer

`churn_probability` takes raw (engineered-`X`) customer rows and returns each model's
predicted churn probability. It runs the whole path a new customer would go through:
engineer the archetype features, project into the trained UMAP embedding, assign the
archetype by the Phase 1 KNN, one-hot it onto the engineered feature matrix, and score.
Everything it touches — `preprocess`, `cluster_prep`, `umap_model`, the archetype KNN,
and the fitted models — was fit on train only.

In [19]:
# The archetype assigner, rebuilt from the Phase 1 train embedding (HDBSCAN has no
# predict, so a new customer's archetype is its nearest neighbours in the embedding).
_arch_knn = KNeighborsClassifier(n_neighbors=15).fit(
    emb_train[clu_train != -1], clu_train[clu_train != -1])


def churn_probability(raw_X):
    """raw_X: rows in the same shape as X_train. Returns a DataFrame of per-model churn
    probabilities (one column per model), plus the assigned archetype."""
    A = archetype_frame(raw_X)
    clu = _arch_knn.predict(umap_model.transform(cluster_prep.transform(A)))
    Xm = np.hstack([to_dense(preprocess.transform(raw_X)),
                    clu_ohe.transform(np.asarray(clu).reshape(-1, 1))])
    out = pd.DataFrame({name: fitted[name].predict_proba(Xm)[:, 1] for name in MODELS},
                       index=raw_X.index)
    out.insert(0, "archetype", [NAMES[c] for c in clu])
    return out


demo = churn_probability(X_test.head(8))
print("Per-customer churn probability (first 8 test customers)\n")
print(demo.to_string(formatters={m: "{:.3f}".format for m in MODELS}))

Per-customer churn probability (first 8 test customers)

        archetype Logistic Regression Decision Tree Random Forest LightGBM XGBoost
803   Archetype 3               0.244         0.000         0.225    0.114   0.064
3549  Archetype 1               0.040         0.046         0.105    0.033   0.030
3515  Archetype 0               0.001         0.000         0.060    0.009   0.011
5162  Archetype 0               0.001         0.000         0.013    0.002   0.002
4642  Archetype 1               0.004         0.000         0.023    0.005   0.006
6096  Archetype 1               0.243         0.356         0.443    0.403   0.395
4504  Archetype 3               0.877         0.716         0.803    0.763   0.805
4067  Archetype 3               0.872         0.837         0.852    0.838   0.834


## Validation — evaluating the five models on the held-out test set

The test customers were assigned their archetype in Phase 1 (`clu_test`, KNN in the
trained embedding) — the test set is never clustered on its own. Each model is scored at
the operating threshold chosen out-of-fold on train, so nothing here is tuned on the data
it is measured on.

Reported per model: the confusion matrix, precision, recall, F1, ROC-AUC and PR-AUC, plus
ROC curves and the distribution of predicted churn probabilities.

In [20]:
rows, cms = [], {}
for name in MODELS:
    p = test_proba_m[name]
    pred = (p >= thr_m[name]).astype(int)
    cms[name] = confusion_matrix(y_te, pred)
    rows.append({
        "model": name, "threshold": thr_m[name],
        "precision": precision_score(y_te, pred),
        "recall": recall_score(y_te, pred),
        "F1": f1_score(y_te, pred),
        "ROC-AUC": roc_auc_score(y_te, p),
        "PR-AUC": average_precision_score(y_te, p),
    })
results = pd.DataFrame(rows).set_index("model").sort_values("F1", ascending=False)

print("=== TEST-set performance (archetype label included as a feature) ===\n")
print(results.to_string(formatters={c: "{:.4f}".format for c in results.columns}))
print(f"\n(Phase 0 baseline, no archetype feature: F1 {BASE_F1:.4f})")
print("\nConfusion matrices  [[TN FP][FN TP]]:")
for name in results.index:
    print(f"  {name:20s} {cms[name].tolist()}")

=== TEST-set performance (archetype label included as a feature) ===

                    threshold precision recall     F1 ROC-AUC PR-AUC
model                                                               
XGBoost                0.3351    0.5525 0.9278 0.6926  0.9166 0.8057
Logistic Regression    0.4299    0.5568 0.9037 0.6891  0.9049 0.7611
LightGBM               0.3600    0.5473 0.9278 0.6885  0.9156 0.8028
Random Forest          0.3749    0.5556 0.8957 0.6858  0.8980 0.7524
Decision Tree          0.4435    0.5612 0.8583 0.6786  0.8793 0.6998

(Phase 0 baseline, no archetype feature: F1 0.7189)

Confusion matrices  [[TN FP][FN TP]]:
  XGBoost              [[754, 281], [27, 347]]
  Logistic Regression  [[766, 269], [36, 338]]
  LightGBM             [[748, 287], [27, 347]]
  Random Forest        [[767, 268], [39, 335]]
  Decision Tree        [[784, 251], [53, 321]]


In [21]:
# Confusion matrices, one small-multiple per model.
from plotly.subplots import make_subplots

order = results.index.tolist()
fig = make_subplots(rows=2, cols=3, subplot_titles=order,
                    horizontal_spacing=0.09, vertical_spacing=0.16)
CM_SCALE = [[0.0, "#f0efec"], [1.0, "#2a78d6"]]
for i, name in enumerate(order):
    r, c = divmod(i, 3)
    cm = cms[name]
    fig.add_trace(go.Heatmap(
        z=cm, x=["pred: stay", "pred: churn"], y=["actual: stay", "actual: churn"],
        colorscale=CM_SCALE, showscale=False, xgap=3, ygap=3,
        text=cm, texttemplate="%{text}", textfont=dict(size=14, color=INK),
        hovertemplate="%{y}, %{x}: %{z}<extra></extra>"), row=r + 1, col=c + 1)

fig.update_yaxes(autorange="reversed", tickfont=dict(color=SECONDARY, size=10),
                 showgrid=False)
fig.update_xaxes(tickfont=dict(color=SECONDARY, size=10), showgrid=False)
for ann in fig.layout.annotations:
    ann.font = dict(color=INK, size=12)
base_layout(fig, "Confusion matrices on the test set", height=560,
            showlegend=False, margin=dict(t=80, r=30, b=40, l=90))
fig.show()

In [22]:
# ROC curves -- the ranking view, independent of any threshold.
fig = go.Figure()
for i, name in enumerate(order):
    fpr, tpr, _ = roc_curve(y_te, test_proba_m[name])
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines", name=f"{name}  (AUC {results.loc[name, 'ROC-AUC']:.3f})",
        line=dict(color=SERIES[i % len(SERIES)], width=2)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", showlegend=False,
                         line=dict(color=DEEMPH, width=1, dash="dash")))
base_layout(fig, "ROC curves — five models on the test set", height=480,
            legend=dict(x=0.98, y=0.02, xanchor="right", yanchor="bottom",
                        font=dict(color=INK), bgcolor="rgba(252,252,251,0.7)"),
            margin=dict(t=60, r=20, b=50, l=60))
fig.update_xaxes(title="False positive rate", **AXIS)
fig.update_yaxes(title="True positive rate", **AXIS)
fig.show()

In [23]:
# Predicted churn score distribution for LightGBM, split by actual outcome.
# The dashed line is this model's own operating threshold (thr_m), not a flat 0.5 --
# that is where LightGBM actually cuts churn from retain in this notebook.
_name = "LightGBM"
_p = test_proba_m[_name]
_y = np.asarray(y_te)
_thr = thr_m[_name]

fig = go.Figure()
for _label, _mask, _color in [("Churned", _y == 1, SERIES[0]),
                              ("Retained", _y == 0, SERIES[1])]:
    fig.add_trace(go.Histogram(
        x=_p[_mask], name=_label, histnorm="percent",
        xbins=dict(start=0.0, end=1.0, size=0.025),
        marker=dict(color=_color, line=dict(color=SURFACE, width=1)), opacity=0.75,
        hovertemplate="<b>%s</b><br>score %%{x}<br>%%{y:.1f}%% of %s<extra></extra>"
                      % (_label, _label.lower())))

fig.add_vline(x=_thr, line=dict(color=INK, width=2, dash="dash"),
              annotation_text=f"threshold {_thr:.3f}", annotation_position="top",
              annotation_font=dict(color=MUTED, size=11))

base_layout(fig, f"{_name} — predicted churn score by actual outcome", height=420,
            barmode="overlay", bargap=0.02,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                        font=dict(color=INK)),
            margin=dict(t=70, r=20, b=50, l=60))
fig.update_xaxes(title="Predicted churn score", range=[0, 1], **AXIS)
fig.update_yaxes(title="% within class", **AXIS)
fig.show()

In [24]:
# Predicted churn score distribution for XGBoost, split by actual outcome.
# The dashed line is this model's own operating threshold (thr_m), not a flat 0.5 --
# that is where XGBoost actually cuts churn from retain in this notebook.
_name = "XGBoost"
_p = test_proba_m[_name]
_y = np.asarray(y_te)
_thr = thr_m[_name]

fig = go.Figure()
for _label, _mask, _color in [("Churned", _y == 1, SERIES[0]),
                              ("Retained", _y == 0, SERIES[1])]:
    fig.add_trace(go.Histogram(
        x=_p[_mask], name=_label, histnorm="percent",
        xbins=dict(start=0.0, end=1.0, size=0.025),
        marker=dict(color=_color, line=dict(color=SURFACE, width=1)), opacity=0.75,
        hovertemplate="<b>%s</b><br>score %%{x}<br>%%{y:.1f}%% of %s<extra></extra>"
                      % (_label, _label.lower())))

fig.add_vline(x=_thr, line=dict(color=INK, width=2, dash="dash"),
              annotation_text=f"threshold {_thr:.3f}", annotation_position="top",
              annotation_font=dict(color=MUTED, size=11))

base_layout(fig, f"{_name} — predicted churn score by actual outcome", height=420,
            barmode="overlay", bargap=0.02,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                        font=dict(color=INK)),
            margin=dict(t=70, r=20, b=50, l=60))
fig.update_xaxes(title="Predicted churn score", range=[0, 1], **AXIS)
fig.update_yaxes(title="% within class", **AXIS)
fig.show()

In [25]:
# --- XGBoost: metrics at adjustable thresholds -----------------------------------
# Edit THRESHOLDS to whatever operating points you want to compare. Each row scores
# XGBoost's held-out test probabilities at that cut: lower threshold -> more customers
# flagged -> higher recall, lower precision. F1 is their harmonic mean.
MODEL = "XGBoost"  # pick one of the five models above
THRESHOLDS = [0.30, 0.40, thr_m[MODEL], 0.50, 0.60, 0.70]   # add/adjust freely

_p = test_proba_m[MODEL]
_y = np.asarray(y_te)
_n = len(_y)

_rows = []
for _t in sorted(set(THRESHOLDS)):
    _pred = (_p >= _t).astype(int)
    _flagged = int(_pred.sum())
    _rows.append({
        "threshold": round(_t, 4),
        "flagged": _flagged,                       # customers predicted to churn
        "flagged_%": _flagged / _n,
        "recall": recall_score(_y, _pred, zero_division=0),
        "precision": precision_score(_y, _pred, zero_division=0),
        "F1": f1_score(_y, _pred, zero_division=0),
    })

xgb_threshold_table = pd.DataFrame(_rows).set_index("threshold")

# Mark the notebook's own tuned operating point so you can see what you're moving from.
xgb_threshold_table["tuned"] = ["<-- current" if abs(t - thr_m[MODEL]) < 1e-9 else ""
                                 for t in xgb_threshold_table.index]

print(f"{MODEL} - recall / precision / F1 by threshold (test set, n={_n}, "
      f"churn rate {_y.mean():.1%})")
print(f"tuned operating threshold = {thr_m[MODEL]:.3f}\n")
xgb_threshold_table.style.format({
    "flagged_%": "{:.1%}", "recall": "{:.3f}",
    "precision": "{:.3f}", "F1": "{:.3f}"})

XGBoost - recall / precision / F1 by threshold (test set, n=1409, churn rate 26.5%)
tuned operating threshold = 0.335



,flagged,flagged_%,recall,precision,F1,tuned
threshold,,,,,,
0.300000,655,46.5%,0.933,0.533,0.678,
0.335100,628,44.6%,0.928,0.553,0.693,
0.400000,581,41.2%,0.882,0.568,0.691,
0.500000,504,35.8%,0.834,0.619,0.711,
0.600000,411,29.2%,0.775,0.706,0.739,
0.700000,350,24.8%,0.695,0.743,0.718,


In [26]:
# --- LightGBM: metrics at adjustable thresholds -----------------------------------
# Edit THRESHOLDS to whatever operating points you want to compare. Each row scores
# LightGBM's held-out test probabilities at that cut: lower threshold -> more customers
# flagged -> higher recall, lower precision. F1 is their harmonic mean.
MODEL = "LightGBM"
THRESHOLDS = [0.30, 0.40, thr_m[MODEL], 0.50, 0.60, 0.70]   # add/adjust freely

_p = test_proba_m[MODEL]
_y = np.asarray(y_te)
_n = len(_y)

_rows = []
for _t in sorted(set(THRESHOLDS)):
    _pred = (_p >= _t).astype(int)
    _flagged = int(_pred.sum())
    _rows.append({
        "threshold": round(_t, 4),
        "flagged": _flagged,                       # customers predicted to churn
        "flagged_%": _flagged / _n,
        "recall": recall_score(_y, _pred, zero_division=0),
        "precision": precision_score(_y, _pred, zero_division=0),
        "F1": f1_score(_y, _pred, zero_division=0),
    })

lgbm_threshold_table = pd.DataFrame(_rows).set_index("threshold")

# Mark the notebook's own tuned operating point so you can see what you're moving from.
lgbm_threshold_table["tuned"] = ["<-- current" if abs(t - thr_m[MODEL]) < 1e-9 else ""
                                 for t in lgbm_threshold_table.index]

print(f"{MODEL} - recall / precision / F1 by threshold (test set, n={_n}, "
      f"churn rate {_y.mean():.1%})")
print(f"tuned operating threshold = {thr_m[MODEL]:.3f}\n")
lgbm_threshold_table.style.format({
    "flagged_%": "{:.1%}", "recall": "{:.3f}",
    "precision": "{:.3f}", "F1": "{:.3f}"})

LightGBM - recall / precision / F1 by threshold (test set, n=1409, churn rate 26.5%)
tuned operating threshold = 0.360



,flagged,flagged_%,recall,precision,F1,tuned
threshold,,,,,,
0.300000,679,48.2%,0.947,0.521,0.672,
0.360000,634,45.0%,0.928,0.547,0.688,
0.400000,606,43.0%,0.912,0.563,0.696,
0.500000,511,36.3%,0.840,0.614,0.710,
0.600000,429,30.4%,0.770,0.671,0.717,
0.700000,332,23.6%,0.676,0.762,0.717,


### Reading the bake-off

- **The metrics table** ranks the five models by F1 at their recall-first operating point,
  and puts the Phase 0 baseline (no archetype feature) next to them. Read whether adding
  the archetype column actually moved F1 — the notebook's standing finding is that it
  will not move it beyond the ±0.01 noise band, for the structural reason in the header.
- **The gradient-boosted pair (LightGBM, XGBoost)** were tuned to push recall at the
  precision floor, so expect them high on recall; the tree/linear baselines are there to
  show how much of the signal is reachable without heavy tuning.
- **The probability distribution** is the more useful artefact for retention: the
  well-separated, right-skewed mass is the pool of high-risk customers to route into the
  SHAP -> LLM outreach layer. A model whose density piles up near the base rate is
  discriminating less, whatever its headline AUC.

## Phase 4 — Why churners leave, and what to do about it (SHAP -> Groq LLM)

Finalises **LightGBM at a churn threshold of 0.40** and explains churn with SHAP, then turns
the drivers into retention actions via a Groq `gpt-oss` model.

**Method choices, made explicit:**

- **No refit.** SHAP runs on the already-fitted `fitted["LightGBM"]` from Phase 3.
- **Cluster removed *after* SHAP, manually.** The archetype one-hot columns are dropped from
  the SHAP matrix. This deletes the archetype's *direct* attribution but not the influence it
  had through the model's split structure; a refit on the no-cluster space would remove that
  too. You chose not to refit (the archetype was a minor feature in Phase 3), and this is the
  honest way to honour it -- the remaining values are still valid SHAP values *for this model*.
- **Original features, not one-hot columns.** SHAP is additive, so each categorical feature's
  attribution is the *sum* of its one-hot children. You read `Internet Type`, not
  `Internet Type_Fiber Optic`.
- **Two populations.** The general SHAP bar explains the customers who *actually* churned
  (`y_te == 1`) -- the analysis view. The LLM advises only on customers the model *predicts*
  will churn (`proba >= 0.40`) -- the action view.

In [27]:
# --- Finalise LightGBM @ 0.40, SHAP on the existing model, drop cluster, collapse to originals
import shap

BEST_MODEL = "LightGBM"
CHURN_THRESHOLD = 0.40
lgbm = fitted[BEST_MODEL]                      # no refit -- the Phase 3 model as-is

# Predictions for the whole test set (used by the LLM targeting and the Excel dump).
test_proba = lgbm.predict_proba(Xte_model)[:, 1]
pred_label = (test_proba >= CHURN_THRESHOLD).astype(int)
print(f"Finalised {BEST_MODEL} @ threshold {CHURN_THRESHOLD}")
print(f"  test recall {recall_score(y_te, pred_label):.3f}  "
      f"precision {precision_score(y_te, pred_label):.3f}  F1 {f1_score(y_te, pred_label):.3f}")

# Leakage guard -- these must never have reached the model's feature space (dropped in cell 3).
_LEAKY = ["Satisfaction Score", "Customer Status", "Churn Label", "Churn Score",
          "CLTV", "Churn Category", "Churn Reason", "Churn Value"]
assert not [f for f in MODEL_FEATURES if any(k in f for k in _LEAKY)], "leaky feature in model"

# SHAP once on the full test set.
explainer = shap.TreeExplainer(lgbm)
_sv = explainer.shap_values(Xte_model)
if isinstance(_sv, list):          # older shap: [class0, class1]
    _sv = _sv[1]
shap_full = np.asarray(_sv)
if shap_full.ndim == 3:            # some versions: (n, features, classes)
    shap_full = shap_full[:, :, 1]
assert shap_full.shape == (len(y_te), len(MODEL_FEATURES))

# 1) drop the archetype/cluster one-hot columns (manual, by name).
_keep = [i for i, f in enumerate(MODEL_FEATURES) if f not in CLUSTER_FEATURES]
_base_names = [MODEL_FEATURES[i] for i in _keep]
shap_base = shap_full[:, _keep]
assert not any("cluster" in f.lower() for f in _base_names), "cluster survived the drop"


# 2) collapse one-hot children back to their original feature (SHAP is additive -> sum).
def encoded_to_original(name):
    body = name.split("__", 1)[1] if "__" in name else name
    if body in num_cols or body in ord_cols:
        return body
    for col in sorted(cat_cols, key=len, reverse=True):
        if body == col or body.startswith(col + "_"):
            return col
    return body


_orig_of = [encoded_to_original(f) for f in _base_names]
_shap_df = pd.DataFrame(shap_base, columns=_orig_of).T.groupby(level=0).sum().T
ORIG_FEATURES = [c for c in (num_cols + ord_cols + cat_cols) if c in _shap_df.columns]
shap_orig = _shap_df[ORIG_FEATURES].values     # (n_test, n_original_features)

assert set(ORIG_FEATURES) <= set(num_cols + ord_cols + cat_cols), "a one-hot name leaked through"
print(f"\nSHAP: {len(MODEL_FEATURES)} model cols "
      f"-> {len(_base_names)} after dropping {len(CLUSTER_FEATURES)} cluster cols "
      f"-> {len(ORIG_FEATURES)} original features")
print(f"predicted churners (proba>={CHURN_THRESHOLD}): {int(pred_label.sum())} of {len(y_te)}")
print(f"actual churners: {int(y_te.sum())}")

Finalised LightGBM @ threshold 0.4
  test recall 0.912  precision 0.563  F1 0.696

SHAP: 45 model cols -> 41 after dropping 4 cluster cols -> 36 original features
predicted churners (proba>=0.4): 606 of 1409
actual churners: 374


In [28]:
# --- General SHAP view: what drove the customers who ACTUALLY churned (mean |SHAP|) ---
TOP_N = 15
_churn = y_te == 1
mean_abs = (pd.Series(np.abs(shap_orig[_churn]).mean(axis=0), index=ORIG_FEATURES)
            .sort_values(ascending=False))
imp = mean_abs.head(TOP_N).sort_values()
_cut = imp.nlargest(3).min()

fig = go.Figure(go.Bar(
    x=imp.values, y=imp.index.tolist(), orientation="h",
    marker=dict(color=SERIES[0], cornerradius=4), width=0.62,
    text=[f"  {v:.3f}" if v >= _cut else "" for v in imp.values],
    textposition="outside", textfont=dict(color=SECONDARY, size=11),
    hovertemplate="<b>%{y}</b><br>mean |SHAP| %{x:.4f}<extra></extra>"))
base_layout(fig, f"LightGBM \u2014 churn drivers across the {int(_churn.sum())} churned "
                 f"customers (mean |SHAP|, original features)", height=470,
            showlegend=False, margin=dict(t=60, r=95, b=50, l=250))
fig.update_xaxes(title="mean |SHAP|  (impact on churn log-odds)",
                 range=[0, imp.max() * 1.18], **AXIS)
fig.update_yaxes(showgrid=False, linecolor=BASELINE, tickfont=dict(color=SECONDARY, size=11))
fig.show()

In [29]:
# --- Feature -> plain-English description (IBM Telco data dictionary) -------------
# Wording from the IBM Telco data dictionary; keyed by ORIGINAL feature name (no one-hot).
FEATURE_DESCRIPTIONS = {
    "Number of Referrals": "How many other people the customer has referred to the company to date.",
    "Tenure in Months": "Total months the customer has been with the company by end of quarter.",
    "Avg Monthly Long Distance Charges": "Average long-distance charges the customer incurs per month.",
    "Avg Monthly GB Download": "Average gigabytes the customer downloads per month.",
    "Monthly Charge": "The customer's current total monthly charge for all their services.",
    "Total Charges": "The customer's total charges, calculated to the end of the quarter.",
    "Total Refunds": "Total refunds the customer has received.",
    "Total Extra Data Charges": "Total charges for data used above the plan allowance.",
    "Total Long Distance Charges": "Total long-distance charges over the customer's tenure.",
    "Age": "The customer's current age, in years.",
    "Number of Dependents": "Number of dependents living with the customer (children, parents, etc.).",
    "Latitude": "Latitude of the customer's primary residence.",
    "Longitude": "Longitude of the customer's primary residence.",
    "Population": "Population of the customer's ZIP-code area.",
    "num_services": "Count of services the customer subscribes to (phone, streaming, security, etc.).",
    "avg_monthly_spend": "Total Charges divided by tenure - average spend per month of tenure.",
    "monthly_to_total_ratio": "Current monthly charge relative to lifetime charges; high for newer customers.",
    "Contract": "The customer's contract term: Month-to-Month, One Year, or Two Year.",
    "tenure_bucket": "Tenure grouped into lifecycle bands (0-3m, 4-12m, 13-24m, 25-48m, 48m+).",
    "Offer": "The last marketing offer the customer accepted (No Offer, Offer A-E).",
    "Phone Service": "Whether the customer subscribes to home phone service (Yes/No).",
    "Multiple Lines": "Whether the customer subscribes to multiple telephone lines (Yes/No).",
    "Internet Type": "Type of internet service: DSL, Fiber Optic, Cable, or No Internet.",
    "Online Security": "Whether the customer subscribes to the add-on online-security service (Yes/No).",
    "Online Backup": "Whether the customer subscribes to the add-on online-backup service (Yes/No).",
    "Device Protection Plan": "Whether the customer has a protection plan for their internet equipment (Yes/No).",
    "Premium Tech Support": "Whether the customer pays for premium tech support with reduced wait times (Yes/No).",
    "Streaming TV": "Whether the customer streams television over their internet service (Yes/No).",
    "Streaming Movies": "Whether the customer streams movies over their internet service (Yes/No).",
    "Streaming Music": "Whether the customer streams music over their internet service (Yes/No).",
    "Unlimited Data": "Whether the customer has an unlimited data plan (Yes/No).",
    "Paperless Billing": "Whether the customer has chosen paperless billing (Yes/No).",
    "Payment Method": "How the customer pays: Bank Withdrawal, Credit Card, or Mailed Check.",
    "Gender": "The customer's gender.",
    "Married": "Whether the customer is married (Yes/No).",
    "has_offer": "Whether the customer accepted any marketing offer (Yes/No).",
}


def describe(feature):
    return FEATURE_DESCRIPTIONS.get(feature, "(no description available)")


_missing = [f for f in ORIG_FEATURES if f not in FEATURE_DESCRIPTIONS]
print(f"description coverage: {len(ORIG_FEATURES) - len(_missing)}/{len(ORIG_FEATURES)} features")
if _missing:
    print("  MISSING:", _missing)

description coverage: 36/36 features


In [30]:
# --- 8a: per-customer retention advice for PREDICTED churners (proba>=0.40), Groq gpt-oss ---
import os

GROQ_MODEL = "openai/gpt-oss-120b"   # gpt-oss on Groq; tiny workload, limits don't bind
N_EXPLAIN = 5                        # how many predicted churners to write up (highest proba)

# Candidate pool = customers the model flags as churn; take the most confident N.
pred_churn_pos = np.where(pred_label == 1)[0]
pred_churn_pos = pred_churn_pos[np.argsort(-test_proba[pred_churn_pos])]   # highest proba first


def _drivers(pos, k=8):
    s = pd.Series(shap_orig[pos], index=ORIG_FEATURES)
    return s.reindex(s.abs().sort_values(ascending=False).index).head(k)


def _peer_context(pos):
    """8a: the customer's archetype + how that segment behaves (context, not a SHAP feature)."""
    c = clu_test[pos]
    name = NAMES.get(c, "Unclustered")
    if name in profile.index:
        return (f"Peer archetype: {name} "
                f"(segment churn rate {profile.loc[name, 'churn_rate']:.0%}, "
                f"{int(profile.loc[name, 'n_train'])} customers in training)")
    return f"Peer archetype: {name}"


def customer_brief(pos):
    raw = X_test.iloc[pos]
    lines = []
    for feat, s in _drivers(pos).items():
        val = raw[feat] if feat in raw.index else ""
        lines.append(f"  - {feat} (this customer: {val})  SHAP {s:+.3f} "
                     f"-> {'raised' if s > 0 else 'lowered'} churn risk\n"
                     f"      meaning: {describe(feat)}")
    return (f"Predicted churner (test index {X_test.index[pos]})\n"
            f"  model churn probability: {test_proba[pos]:.2f} (threshold {CHURN_THRESHOLD})\n"
            f"  {_peer_context(pos)}\n"
            f"  top SHAP drivers (signed, in churn log-odds):\n" + "\n".join(lines))


SYSTEM = (
    "You are a telecom retention analyst. You are given one customer the churn model flags as "
    "likely to churn, with SHAP attributions (each driver's value, signed SHAP contribution, "
    "plain-English meaning) and the customer's behavioural archetype for peer context.\n"
    "Write two short sections:\n"
    "1. WHY THEY ARE AT RISK - 2-4 bullets, each naming a specific driver and what about this "
    "customer's value pushes risk up; use the archetype to frame the peer pattern.\n"
    "2. ACTIONS TO PREVENT CHURN - 2-4 concrete retention moves tied to those drivers "
    "(offer, plan change, outreach), each naming the driver it addresses and tailored to the "
    "archetype.\n"
    "SHAP is association, not proof of cause: say what the pattern suggests, don't assert cause. "
    "No preamble."
)

api_key = os.environ.get("GROQ_API_KEY")
llm_outputs = {}                     # test-index -> recommendation text

if not api_key:
    print("GROQ_API_KEY not set - skipping LLM calls. Example brief the model would receive:\n")
    if len(pred_churn_pos):
        print(customer_brief(int(pred_churn_pos[0])))
    print("\nSet the key and re-run:  export GROQ_API_KEY='gsk_...'")
else:
    from groq import Groq
    client = Groq(api_key=api_key)
    for pos in pred_churn_pos[:N_EXPLAIN]:
        pos = int(pos)
        brief = customer_brief(pos)
        resp = client.chat.completions.create(
            model=GROQ_MODEL, max_completion_tokens=750,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": brief}])
        llm_outputs[X_test.index[pos]] = resp.choices[0].message.content
        print("=" * 80)
        print(brief)
        print("-" * 80)
        print(resp.choices[0].message.content)
        print()

Predicted churner (test index 18)
  model churn probability: 0.99 (threshold 0.4)
  Peer archetype: Archetype 3 (segment churn rate 54%, 920 customers in training)
  top SHAP drivers (signed, in churn log-odds):
  - Contract (this customer: Month-to-Month)  SHAP +1.456 -> raised churn risk
      meaning: The customer's contract term: Month-to-Month, One Year, or Two Year.
  - Latitude (this customer: 32.85723)  SHAP +0.863 -> raised churn risk
      meaning: Latitude of the customer's primary residence.
  - Age (this customer: 74)  SHAP +0.593 -> raised churn risk
      meaning: The customer's current age, in years.
  - Number of Referrals (this customer: 0)  SHAP +0.502 -> raised churn risk
      meaning: How many other people the customer has referred to the company to date.
  - Longitude (this customer: -117.209774)  SHAP +0.388 -> raised churn risk
      meaning: Longitude of the customer's primary residence.
  - Number of Dependents (this customer: 0)  SHAP +0.302 -> raised churn 

In [31]:
# --- 8b: one reusable retention playbook per archetype (predicted churners in each) ---
def archetype_brief(c):
    name = NAMES[c]
    seg = (clu_test == c) & (pred_label == 1)
    n = int(seg.sum())
    drivers = pd.Series(shap_orig[seg].mean(axis=0), index=ORIG_FEATURES)
    drivers = drivers.reindex(drivers.abs().sort_values(ascending=False).index).head(8)
    lines = [f"  - {f}: mean SHAP {v:+.3f} "
             f"({'raises' if v > 0 else 'lowers'} churn) | {describe(f)}"
             for f, v in drivers.items()]
    seg_churn = profile.loc[name, "churn_rate"] if name in profile.index else float("nan")
    return (f"Archetype: {name}\n"
            f"  predicted churners in this segment: {n}\n"
            f"  segment training churn rate: {seg_churn:.0%}\n"
            f"  segment mean SHAP drivers (signed):\n" + "\n".join(lines))


PLAYBOOK_SYSTEM = (
    "You are a telecom retention strategist. You are given one behavioural archetype (customer "
    "segment) and the churn model's average SHAP drivers for the customers in it the model "
    "flags as likely to churn. Produce a concise retention PLAYBOOK for this segment:\n"
    "1. SEGMENT PROFILE - 1-2 sentences on what defines this segment's churn risk.\n"
    "2. PLAYBOOK - 3-5 prioritised, reusable retention plays, each tied to a named driver, "
    "specifying the trigger, the offer/action, and the risk of the play.\n"
    "SHAP is association, not proof of cause. No preamble."
)

archetype_playbooks = {}
if not api_key:
    print("GROQ_API_KEY not set - skipping playbooks. Example segment brief:\n")
    print(archetype_brief(ARCHETYPES[0]))
else:
    for c in ARCHETYPES:
        if ((clu_test == c) & (pred_label == 1)).sum() == 0:
            continue
        brief = archetype_brief(c)
        resp = client.chat.completions.create(
            model=GROQ_MODEL, max_completion_tokens=800,
            messages=[{"role": "system", "content": PLAYBOOK_SYSTEM},
                      {"role": "user", "content": brief}])
        archetype_playbooks[NAMES[c]] = resp.choices[0].message.content
        print("=" * 80)
        print(f"RETENTION PLAYBOOK - {NAMES[c]}")
        print("=" * 80)
        print(resp.choices[0].message.content)
        print()

RETENTION PLAYBOOK - Archetype 0
**SEGMENT PROFILE**  
Customers in Archetype 0 are early‑stage, month‑to‑month users who have low tenure, a high proportion of recent spend to lifetime spend, and modest long‑distance usage; these factors collectively lift their churn propensity despite relatively high data consumption and monthly bill size.

**PLAYBOOK**

| # | Driver (SHAP) | Trigger | Offer / Action | Risk |
|---|---------------|---------|----------------|------|
| 1 | **Contract** ( +1.146 ) | Contract = Month‑to‑Month **and** tenure < 12 mo | Auto‑upgrade to a 12‑month plan with a 20 % discount on the first 3 months + a “price‑lock” guarantee | Revenue dilution; may attract price‑sensitive users who still leave after discount period |
| 2 | **monthly_to_total_ratio** ( +0.364 ) | Ratio > 0.80 (i.e., >80 % of lifetime spend incurred in the last 3 months) | “Welcome‑back” bundle: add 5 GB extra data + free streaming for 30 days, tied to a 12‑month commitment | Increased ARPU pressure

In [32]:
# --- Dump the entire test set (churn + non-churn) to Excel ------------------------
OUT_XLSX = "telco_test_predictions.xlsx"

# Recover the Customer ID that cell 3 dropped, aligned by the test set's original index.
_customer_id = pd.read_csv(DATA_PATH)["Customer ID"].loc[X_test.index].values

# Top-3 signed SHAP drivers per customer, as a readable string.
def _top3(pos):
    s = pd.Series(shap_orig[pos], index=ORIG_FEATURES)
    s = s.reindex(s.abs().sort_values(ascending=False).index).head(3)
    return "; ".join(f"{f} ({v:+.2f})" for f, v in s.items())

dump = pd.DataFrame({
    "Customer ID": _customer_id,
    "archetype": [NAMES.get(c, "Unclustered") for c in clu_test],
    "churn_probability": np.round(test_proba, 4),
    "predicted_churn_label": pred_label,
    "actual_churn_value": y_te,
    "top_churn_drivers": [_top3(i) for i in range(len(y_te))],
    "llm_suggestion": [llm_outputs.get(X_test.index[i], "") for i in range(len(y_te))],
}, index=X_test.index)

dump.to_excel(OUT_XLSX, index_label="row_index")
print(f"Wrote {OUT_XLSX}: {len(dump)} rows "
      f"({int(pred_label.sum())} predicted churn, {int((pred_label == 0).sum())} predicted retain)")
print(f"LLM suggestions filled for {sum(bool(v) for v in dump['llm_suggestion'])} customers")
dump.head(8)

Wrote telco_test_predictions.xlsx: 1409 rows (606 predicted churn, 803 predicted retain)
LLM suggestions filled for 5 customers


,Customer ID,archetype,churn_probability,predicted_churn_label,actual_churn_value,top_churn_drivers,llm_suggestion
803,5019-GQVCR,Archetype 3,0.1139,0,0,Contract (-1.91); Number of Referrals (+0.52);...,
3549,3766-EJLFL,Archetype 1,0.0328,0,0,Number of Dependents (-1.66); Contract (-1.43)...,
3515,9764-REAFF,Archetype 0,0.0092,0,0,Number of Referrals (-1.82); Contract (-1.09);...,
5162,6199-IPCAO,Archetype 0,0.0016,0,0,Number of Referrals (-1.76); Contract (-1.26);...,
4642,2378-YIZKA,Archetype 1,0.0046,0,0,Number of Referrals (-1.57); Contract (-1.11);...,
6096,7964-VEXDG,Archetype 1,0.4031,1,0,Contract (+1.41); Number of Dependents (-0.78)...,
4504,1666-JXLKU,Archetype 3,0.7630,1,0,Contract (+1.04); Number of Referrals (+0.47);...,
4067,6734-GMPVK,Archetype 3,0.8379,1,0,Contract (+1.06); Number of Referrals (+0.42);...,
